# Travel Reimbursement App

This notebook contains the full local implementation of the travel reimbursement workflow described in the PRD.

**Use case:** Employee submits a claim, manager reviews it, finance marks it as paid.

In [4]:
from __future__ import annotations

from datetime import datetime, timezone
from typing import Literal
from uuid import uuid4

from fastapi import FastAPI, HTTPException, Query
from pydantic import BaseModel, Field

ALLOWED_STATUSES = ["Draft", "Submitted", "Under Review", "Approved", "Rejected", "Paid"]
ALLOWED_ROLES = ["employee", "manager", "finance"]

app = FastAPI(
    title="Travel Reimbursement Manager",
    version="1.0.0",
    description="End-to-end local app for employee travel reimbursement processing.",
)


class ExpenseItem(BaseModel):
    category: str = Field(..., min_length=1)
    amount: float = Field(..., gt=0)
    description: str = ""


class ClaimCreate(BaseModel):
    employee_name: str = Field(..., min_length=1)
    purpose: str = Field(..., min_length=1)
    destination: str = Field(..., min_length=1)
    start_date: str
    end_date: str
    expense_items: list[ExpenseItem] = Field(..., min_length=1)
    receipt_documents: list[str] = Field(default_factory=list)
    comments: str = ""


class StatusUpdate(BaseModel):
    status: Literal["Draft", "Submitted", "Under Review", "Approved", "Rejected", "Paid"]
    actor_role: Literal["employee", "manager", "finance"]
    actor_name: str = Field(..., min_length=1)
    comment: str = ""


class AuditEntry(BaseModel):
    timestamp: str
    actor_name: str
    actor_role: str
    action: str
    comment: str = ""


class Claim(BaseModel):
    id: str
    employee_name: str
    purpose: str
    destination: str
    start_date: str
    end_date: str
    expense_items: list[ExpenseItem]
    receipt_documents: list[str] = Field(default_factory=list)
    comments: str = ""
    status: Literal["Draft", "Submitted", "Under Review", "Approved", "Rejected", "Paid"]
    total_amount: float
    audit_log: list[AuditEntry] = Field(default_factory=list)


claims_db: dict[str, Claim] = {}


def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()


def compute_total(items: list[ExpenseItem]) -> float:
    return round(sum(item.amount for item in items), 2)


def append_audit(claim: Claim, actor_name: str, actor_role: str, action: str, comment: str = "") -> None:
    claim.audit_log.append(
        AuditEntry(
            timestamp=utc_now(),
            actor_name=actor_name,
            actor_role=actor_role,
            action=action,
            comment=comment,
        )
    )


def allowed_transitions() -> dict[str, list[str]]:
    return {
        "Draft": ["Submitted", "Rejected"],
        "Submitted": ["Under Review", "Rejected"],
        "Under Review": ["Approved", "Rejected"],
        "Approved": ["Paid"],
        "Rejected": [],
        "Paid": [],
    }


@app.get("/health")
def health_check() -> dict:
    return {
        "status": "ok",
        "app": "travel-reimbursement-manager",
        "version": app.version,
        "totals": {
            "claims": len(claims_db),
            "pending": sum(1 for claim in claims_db.values() if claim.status not in {"Rejected", "Paid"}),
        },
    }


@app.get("/api/claims")
def list_claims(
    employee_name: str | None = Query(default=None),
    status: str | None = Query(default=None),
) -> dict:
    data = list(claims_db.values())
    if employee_name:
        data = [claim for claim in data if claim.employee_name.lower() == employee_name.lower()]
    if status:
        data = [claim for claim in data if claim.status.lower() == status.lower()]
    return {"claims": data}


@app.post("/api/claims", status_code=201)
def create_claim(payload: ClaimCreate) -> Claim:
    if not payload.expense_items:
        raise HTTPException(status_code=400, detail="At least one expense item is required")
    if payload.end_date < payload.start_date:
        raise HTTPException(status_code=400, detail="End date must be after or equal to start date")

    claim_id = f"CLAIM-{uuid4().hex[:8].upper()}"
    claim = Claim(
        id=claim_id,
        employee_name=payload.employee_name,
        purpose=payload.purpose,
        destination=payload.destination,
        start_date=payload.start_date,
        end_date=payload.end_date,
        expense_items=payload.expense_items,
        receipt_documents=payload.receipt_documents,
        comments=payload.comments,
        status="Submitted",
        total_amount=compute_total(payload.expense_items),
    )
    append_audit(
        claim,
        actor_name=payload.employee_name,
        actor_role="employee",
        action="Claim submitted",
        comment="Employee submitted travel reimbursement request",
    )
    claims_db[claim_id] = claim
    return claim


@app.get("/api/claims/{claim_id}")
def get_claim(claim_id: str) -> Claim:
    claim = claims_db.get(claim_id)
    if not claim:
        raise HTTPException(status_code=404, detail="Claim not found")
    return claim


@app.get("/api/claims/employee/{employee_name}")
def get_claims_by_employee(employee_name: str) -> dict:
    matches = [claim for claim in claims_db.values() if claim.employee_name.lower() == employee_name.lower()]
    return {"employee_name": employee_name, "claims": matches}


@app.patch("/api/claims/{claim_id}/status")
def update_claim_status(claim_id: str, payload: StatusUpdate) -> Claim:
    claim = claims_db.get(claim_id)
    if not claim:
        raise HTTPException(status_code=404, detail="Claim not found")
    if payload.actor_role not in ALLOWED_ROLES:
        raise HTTPException(status_code=400, detail="Invalid actor role")

    valid_transitions = allowed_transitions().get(claim.status, [])
    if payload.status not in valid_transitions:
        raise HTTPException(
            status_code=400,
            detail=(
                f"Status transition from '{claim.status}' to '{payload.status}' is not allowed "
                f"for role '{payload.actor_role}'"
            ),
        )

    if payload.actor_role == "employee" and payload.status not in {"Draft", "Submitted"}:
        raise HTTPException(status_code=403, detail="Employees cannot approve or pay claims")
    if payload.actor_role == "manager" and payload.status not in {"Under Review", "Approved", "Rejected"}:
        raise HTTPException(status_code=403, detail="Managers can only review, approve, or reject claims")
    if payload.actor_role == "finance" and payload.status != "Paid":
        raise HTTPException(status_code=403, detail="Finance can only mark claims as paid")

    previous_status = claim.status
    claim.status = payload.status
    append_audit(
        claim,
        actor_name=payload.actor_name,
        actor_role=payload.actor_role,
        action=f"Status changed: {previous_status} -> {payload.status}",
        comment=payload.comment,
    )
    return claim


@app.post("/api/claims/{claim_id}/documents")
def add_receipt_documents(claim_id: str, document_names: list[str]) -> Claim:
    claim = claims_db.get(claim_id)
    if not claim:
        raise HTTPException(status_code=404, detail="Claim not found")
    claim.receipt_documents.extend(document_names)
    append_audit(
        claim,
        actor_name="system",
        actor_role="employee",
        action="Receipt documents attached",
        comment=f"Added {len(document_names)} document(s)",
    )
    return claim


@app.get("/api/dashboard")
def dashboard() -> dict:
    return {
        "total_claims": len(claims_db),
        "submitted": sum(1 for claim in claims_db.values() if claim.status == "Submitted"),
        "approved": sum(1 for claim in claims_db.values() if claim.status == "Approved"),
        "paid": sum(1 for claim in claims_db.values() if claim.status == "Paid"),
        "rejected": sum(1 for claim in claims_db.values() if claim.status == "Rejected"),
        "total_reimbursable_amount": round(sum(claim.total_amount for claim in claims_db.values()), 2),
    }


# Notebook-safe usage: use TestClient for validation in Jupyter.
# Do not start uvicorn inside a notebook because it can trigger a reload loop.


In [5]:
from fastapi.testclient import TestClient

client = TestClient(app)

health = client.get('/health')
print('HEALTH', health.status_code, health.json())

claim = client.post('/api/claims', json={
    'employee_name': 'Alice',
    'purpose': 'Conference travel',
    'destination': 'Boston',
    'start_date': '2026-09-10',
    'end_date': '2026-09-12',
    'expense_items': [
        {'category': 'Flight', 'amount': 250.0, 'description': 'Round trip'},
        {'category': 'Hotel', 'amount': 400.0, 'description': '2 nights'}
    ],
    'receipt_documents': ['flight.pdf', 'hotel.pdf'],
    'comments': 'Team travel reimbursement'
})
print('CREATE', claim.status_code, claim.json()['status'], claim.json()['total_amount'])

claim_id = claim.json()['id']
review = client.patch(f'/api/claims/{claim_id}/status', json={
    'status': 'Under Review',
    'actor_role': 'manager',
    'actor_name': 'Sam Manager',
    'comment': 'Reviewing receipts and travel details'
})
print('REVIEW', review.status_code, review.json()['status'])

approved = client.patch(f'/api/claims/{claim_id}/status', json={
    'status': 'Approved',
    'actor_role': 'manager',
    'actor_name': 'Sam Manager',
    'comment': 'Approved after verification'
})
print('APPROVED', approved.status_code, approved.json()['status'])

paid = client.patch(f'/api/claims/{claim_id}/status', json={
    'status': 'Paid',
    'actor_role': 'finance',
    'actor_name': 'Finance Team',
    'comment': 'Settlement processed'
})
print('PAID', paid.status_code, paid.json()['status'])

print('DASHBOARD', client.get('/api/dashboard').status_code)

HEALTH 200 {'status': 'ok', 'app': 'travel-reimbursement-manager', 'version': '1.0.0', 'totals': {'claims': 0, 'pending': 0}}
CREATE 201 Submitted 650.0
REVIEW 200 Under Review
APPROVED 200 Approved
PAID 200 Paid
DASHBOARD 200
